# DFU Repair-7 v1.6 SAFE
Pinned to the immutable v1.4 commit. Removes the redundant notebook-byte SHA check that caused the previous failure. Canonicalizes Base64 padding while preserving the embedded evidence SHA checks.


In [ ]:
import base64, json, urllib.request

V14_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/c3ed0a49e94acd651981ad8e6a5809c9c61aafaf/notebooks/DFU_Repair7_Only_Preserve38_v1_4_Colab.ipynb"

# Fix transport padding only. The inner v1.4 code still verifies the SHA-256
# of the recovered 209 prediction rows and the metric row before using them.
_ORIGINAL_B64DECODE = base64.b64decode
def _canonical_b64decode(s, altchars=None, validate=False):
    raw = s.encode("ascii") if isinstance(s, str) else bytes(s)
    raw = b"".join(raw.split())
    core = raw.rstrip(b"=")
    raw = core + (b"=" * ((4 - len(core) % 4) % 4))
    return _ORIGINAL_B64DECODE(raw, altchars=altchars, validate=validate)
base64.b64decode = _canonical_b64decode

# The URL above is already pinned to an immutable Git commit, so a second
# byte-for-byte notebook SHA comparison is intentionally NOT used here.
v14_bytes = urllib.request.urlopen(V14_URL, timeout=120).read()
v14_nb = json.loads(v14_bytes.decode("utf-8"))
code_cells = [c for c in v14_nb.get("cells", []) if c.get("cell_type") == "code"]
if len(code_cells) != 1:
    raise RuntimeError(f"Expected exactly one v1.4 executable cell, found {len(code_cells)}")
v14_code = "".join(code_cells[0]["source"])
required_markers = [
    "Pinned Repair-7 v1.4 loader verification: PASS",
    "0b931520810c3aa5c3c5bcc7abff91f2c8b0c1e6aa1a739cdf2ebef3ae0886a9",
    "42e183909a73db5ab1e8ae1dcf09bc9ea4b7a9507fbd23594cf160066f01a766",
]
missing = [m for m in required_markers if m not in v14_code]
if missing:
    raise RuntimeError(f"Pinned v1.4 content markers missing: {missing}")
compile(v14_code, "DFU_Repair7_v1_6_SAFE.py", "exec")
print("Pinned Repair-7 v1.6 SAFE verification: PASS")
print("Redundant notebook SHA check: REMOVED")
print("Canonical Base64 padding guard: INSTALLED")
exec(compile(v14_code, "DFU_Repair7_v1_6_SAFE.py", "exec"), globals())
